In [1]:
import requests
import zipfile
import pandas as pd
import io
import os
import tempfile

In [2]:
def process_backblaze_data(zip_url):
    """
    Downloads a zip file from Backblaze, extracts CSVs, and concatenates them into a single DataFrame.
    """
    print(f"Downloading {zip_url}...")
    response = requests.get(zip_url)
    response.raise_for_status() # Raise an exception for HTTP errors

    # Create a temporary directory to extract files
    with tempfile.TemporaryDirectory() as temp_dir:
        print(f"Extracting files to temporary directory: {temp_dir}")

        # Read the content into a BytesIO object and extract
        with zipfile.ZipFile(io.BytesIO(response.content)) as zf:
            zf.extractall(temp_dir)

        print("Extraction complete. Identifying and concatenating CSV files...")

        all_csv_files = []
        for root, _, files in os.walk(temp_dir):
            for file in files:
                file_path = os.path.join(root, file)
                # Exclude files from __MACOSX directory and files starting with ._
                if file_path.endswith('.csv') and '__MACOSX' not in file_path and not file.startswith('._'):
                    all_csv_files.append(file_path)

        if not all_csv_files:
            print("No CSV files found in the extracted content.")
            return pd.DataFrame() # Return empty DataFrame if no CSVs found
        else:
            # Read and concatenate all CSV files
            df_list = []
            for csv_file in all_csv_files:
                print(f"Reading {csv_file}...")
                df_list.append(pd.read_csv(csv_file))

            combined_df = pd.concat(df_list, ignore_index=True)
            print("All CSV files concatenated successfully.")
            return combined_df

In [3]:
# URL for Q4 2017 data
zip_url_q4 = "https://f001.backblazeb2.com/file/Backblaze-Hard-Drive-Data/data_Q4_2017.zip"

# Process the Q4 data using the function
print("\n--- Processing Q4 2017 Data ---")
combined_df_q4 = process_backblaze_data(zip_url_q4)



--- Processing Q4 2017 Data ---
Extracting files to temporary directory: /tmp/tmpbnhq3esg
Extraction complete. Identifying and concatenating CSV files...
Reading /tmp/tmpbnhq3esg/2017-11-21.csv...
Reading /tmp/tmpbnhq3esg/2017-11-13.csv...
Reading /tmp/tmpbnhq3esg/2017-10-20.csv...
Reading /tmp/tmpbnhq3esg/2017-10-19.csv...
Reading /tmp/tmpbnhq3esg/2017-10-14.csv...
Reading /tmp/tmpbnhq3esg/2017-10-16.csv...
Reading /tmp/tmpbnhq3esg/2017-10-24.csv...
Reading /tmp/tmpbnhq3esg/2017-10-05.csv...
Reading /tmp/tmpbnhq3esg/2017-11-05.csv...
Reading /tmp/tmpbnhq3esg/2017-11-08.csv...
Reading /tmp/tmpbnhq3esg/2017-11-29.csv...
Reading /tmp/tmpbnhq3esg/2017-11-11.csv...
Reading /tmp/tmpbnhq3esg/2017-10-12.csv...
Reading /tmp/tmpbnhq3esg/2017-10-30.csv...
Reading /tmp/tmpbnhq3esg/2017-11-06.csv...
Reading /tmp/tmpbnhq3esg/2017-12-19.csv...
Reading /tmp/tmpbnhq3esg/2017-12-06.csv...
Reading /tmp/tmpbnhq3esg/2017-12-09.csv...
Reading /tmp/tmpbnhq3esg/2017-10-03.csv...
Reading /tmp/tmpbnhq3esg/201

In [4]:
combined_df_q4.shape

(8266213, 95)

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
file_path = '/content/drive/MyDrive/Colab Notebooks/modele stocastique/full_combined_df.csv'
loaded_df = pd.read_csv(file_path)
print(f"DataFrame loaded successfully. Shape: {loaded_df.shape}")

DataFrame loaded successfully. Shape: (14200734, 95)


In [7]:
full_combined_df = pd.concat([loaded_df, combined_df_q4], ignore_index=True)
print(f"Shape of the concatenated DataFrame: {full_combined_df.shape}")

Shape of the concatenated DataFrame: (22466947, 95)


In [9]:
output_dir = '/content/drive/MyDrive/Colab Notebooks/modele stocastique'
output_file_path = os.path.join(output_dir, 'full_combined_df_final.csv')

os.makedirs(output_dir, exist_ok=True)
full_combined_df.to_csv(output_file_path, index=False)

print(f"DataFrame successfully saved to: {output_file_path}")

DataFrame successfully saved to: /content/drive/MyDrive/Colab Notebooks/modele stocastique/full_combined_df_final.csv
